In [8]:
import tabula
import pandas as pd

# Path to your document
file_path = r"G:\My Drive\00) Interview Prep\00) Companies\EFG\Round 3 Slides\stream_2762463_427c397b-57d2-4955-962e-8389c43556e1_doc1.pdf"

# Read all tables from all pages
# 'pages="all"' ensures the entire 65+ page document is processed
tables = tabula.read_pdf(file_path, pages='all', multiple_tables=True)

# Combine all extracted tables into a single DataFrame
df = pd.concat(tables, ignore_index=True)

# Save to CSV or Excel
df.to_csv("ownership_data.csv", index=False)

In [46]:
def simple_extract(file_path):
    print("Starting extraction... this may take a minute for 65+ pages.")
    
    tables = tabula.read_pdf(file_path, pages='all')
    
    if not tables:
        print("Error: Tabula could not find any tables.")
        return None

    # Merge all pages into one DataFrame
    df = pd.concat(tables, ignore_index=True)

    # Cleanup: Strip any weird whitespace from column names
    df.columns = [str(c).strip() for c in df.columns]

    # Numeric Cleaning: Convert columns to numbers safely
    # This handles the dots (thousands) and commas (decimals)
    numeric_cols = ['HOLDINGS_SCRIPLESS', 'HOLDINGS_SCRIP', 'TOTAL_HOLDING_SHARES', 'PERCENTAGE']
    
    for col in numeric_cols:
        if col in df.columns:
            # We convert to string, remove dots, change commas to dots, then force to numeric
            df[col] = (
                df[col].astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

# --- RUN IT ---
file_path = r"G:\My Drive\00) Interview Prep\00) Companies\EFG\Round 3 Slides\stream_2762463_427c397b-57d2-4955-962e-8389c43556e1_doc1.pdf"
df = simple_extract(file_path)

if df is not None:
    # Remove rows where the main data is missing (handles footer/header noise)
    df = df.dropna(subset=[df.columns[0]]) 
    
    df.to_csv("cleaned_ownership_data.csv", index=False)
    print(f"Success! {len(df)} rows saved to cleaned_ownership_data.csv")
    print("\nColumn names found in your PDF:")
    print(df.columns.tolist())

Starting extraction... this may take a minute for 65+ pages.
Success! 7257 rows saved to cleaned_ownership_data.csv

Column names found in your PDF:
['DATE', 'SHARE_CODE', 'ISSUER_NAME', 'INVESTOR_NAME', 'INVESTOR_TYPE', 'LOCAL_FOREIGN', 'NATIONALITY', 'DOMICILE', 'HOLDINGS_SCRIPLESS', 'HOLDINGS_SCRIP', 'TOTAL_HOLDING_SHARES', 'PERCENTAGE']


In [47]:
def analyze_foreign_ownership(file_path):
    print(f"Loading data from {file_path}...\n")
    df = pd.read_csv(file_path)

    # Note: Adjust these column names if your CSV headers differ slightly
    domicile_col = 'DOMICILE'  # The column indicating country
    investor_col = 'INVESTOR NAME'
    issuer_col = 'ISSUER NAME' # Or 'SHARE CODE' depending on your header
    percent_col = 'PERCENTAGE'
    shares_col = 'TOTAL HOLDING SHARES'

    # Clean the domicile column for accurate filtering (uppercase, remove blanks)
    if domicile_col in df.columns:
        df['DOMICILE_CLEAN'] = df[domicile_col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    else:
        print(f"Error: Column '{domicile_col}' not found. Please update the column name.")
        return

    # Filter out domestic holdings (anything containing 'INDONESIA')
    foreign_df = df[~df['DOMICILE_CLEAN'].str.contains('INDONESIA')]

    # --- INSIGHT 1: Top Foreign Jurisdictions ---
    # Which countries house the most >1% investors?
    top_domiciles = foreign_df['DOMICILE_CLEAN'].value_counts().head(5)
    
    print("--- TOP 5 FOREIGN DOMICILES (By Number of >1% Holdings) ---")
    print(top_domiciles.to_string())
    print("\n")

    # --- INSIGHT 2: Largest Offshore Holdings ---
    # Find the largest foreign positions by percentage ownership
    if percent_col in foreign_df.columns:
        top_investments = foreign_df.sort_values(by=percent_col, ascending=False).head(5)
        
        print("--- TOP 5 LARGEST FOREIGN BLOCK HOLDINGS ---")
        # Select specific columns to display clearly
        cols_to_show = [investor_col, issuer_col, 'DOMICILE_CLEAN', percent_col, shares_col]
        
        # Filter to only include columns that actually exist in the dataframe
        cols_to_show = [c for c in cols_to_show if c in foreign_df.columns]
        
        print(top_investments[cols_to_show].to_string(index=False))
    else:
        print(f"Percentage column '{percent_col}' not found for sorting.")

# Run the analysis
analyze_foreign_ownership("cleaned_ownership_data.csv")

Loading data from cleaned_ownership_data.csv...

--- TOP 5 FOREIGN DOMICILES (By Number of >1% Holdings) ---
DOMICILE_CLEAN
SINGAPORE                  656
UNKNOWN                    584
VIRGIN ISLANDS, BRITISH    193
HONG KONG                   85
MALAYSIA                    82


--- TOP 5 LARGEST FOREIGN BLOCK HOLDINGS ---
DOMICILE_CLEAN  PERCENTAGE
     HONG KONG       99.35
 UNITED STATES       98.79
     SINGAPORE       95.92
   SWITZERLAND       95.50
     HONG KONG       92.54


In [48]:
import pandas as pd

def analyze_ownership_proportions(df):
    # --- 1. CLEANING STEP (Fixing the String to Numeric Error) ---
    # Remove dots (thousands separators) and change commas to dots (decimals)
    df['TOTAL_HOLDING_SHARES'] = (
        df['TOTAL_HOLDING_SHARES']
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    # Convert the cleaned strings into actual numbers
    df['TOTAL_HOLDING_SHARES'] = pd.to_numeric(df['TOTAL_HOLDING_SHARES'], errors='coerce')
    
    # --- 2. CLASSIFICATION ---
    # Ensure DOMICILE column is clean
    df['DOMICILE_CLEAN'] = df['DOMICILE'].fillna('UNKNOWN').astype(str).str.upper().str.strip()

    # Create classification
    df['GEO_CLASSIFICATION'] = df['DOMICILE_CLEAN'].apply(
        lambda x: 'Domestic (Indonesia)' if 'INDONESIA' in x else 'Foreign'
    )

    # --- 3. CALCULATIONS ---
    # Proportion by Number of Block Holdings (>1% positions)
    holdings_count = df['GEO_CLASSIFICATION'].value_counts()
    holdings_prop = df['GEO_CLASSIFICATION'].value_counts(normalize=True) * 100

    print("--- PROPORTION BY NUMBER OF >1% POSITIONS ---")
    for category in holdings_count.index:
        print(f"{category}: {holdings_count[category]} positions ({holdings_prop[category]:.2f}%)")
    print("\n")

    # Proportion by Total Share Volume
    volume_sum = df.groupby('GEO_CLASSIFICATION')['TOTAL_HOLDING_SHARES'].sum()
    volume_prop = (volume_sum / volume_sum.sum()) * 100

    print("--- PROPORTION BY TOTAL SHARE VOLUME ---")
    for category in volume_sum.index:
        print(f"{category}: {volume_sum[category]:,.0f} shares ({volume_prop[category]:.2f}%)")

df = pd.read_csv("cleaned_ownership_data.csv")
analyze_ownership_proportions(df)

--- PROPORTION BY NUMBER OF >1% POSITIONS ---
Domestic (Indonesia): 5149 positions (70.95%)
Foreign: 2108 positions (29.05%)


--- PROPORTION BY TOTAL SHARE VOLUME ---
Domestic (Indonesia): 5,653,469,695,474 shares (59.10%)
Foreign: 3,911,948,175,068 shares (40.90%)


In [49]:
def analyze_scrip_ratio(df):
    # Fill NAs with 0 to safely calculate totals
    df['HOLDINGS_SCRIP'] = df['HOLDINGS_SCRIP'].fillna(0)
    df['HOLDINGS_SCRIPLESS'] = df['HOLDINGS_SCRIPLESS'].fillna(0)
    
    # Calculate company-wide scrip vs scripless totals
    issuer_scrip = df.groupby(['SHARE_CODE', 'ISSUER_NAME'])[['HOLDINGS_SCRIP', 'TOTAL_HOLDING_SHARES']].sum().reset_index()
    
    # Calculate the percentage of physical (scrip) shares for each company
    issuer_scrip['SCRIP_PERCENTAGE'] = (issuer_scrip['HOLDINGS_SCRIP'] / issuer_scrip['TOTAL_HOLDING_SHARES']) * 100
    
    # Filter for companies that actually have scrip shares, then sort
    legacy_companies = issuer_scrip[issuer_scrip['SCRIP_PERCENTAGE'] > 0].sort_values(by='SCRIP_PERCENTAGE', ascending=False)
    
    print("--- TOP 5 ISSUERS WITH HIGHEST PHYSICAL SCRIP RATIO (Legacy Structures) ---")
    print(legacy_companies[['SHARE_CODE', 'ISSUER_NAME', 'SCRIP_PERCENTAGE']].head().to_string(index=False))

analyze_scrip_ratio(df)

--- TOP 5 ISSUERS WITH HIGHEST PHYSICAL SCRIP RATIO (Legacy Structures) ---
SHARE_CODE                   ISSUER_NAME  SCRIP_PERCENTAGE
      ADCP    ADHI COMMUTER PROPERTI Tbk             100.0
      PTPS               PULAU SUBUR Tbk             100.0
      WIKA    WIJAYA KARYA (PERSERO) Tbk             100.0
      HMSP HANJAYA MANDALA SAMPOERNA Tbk             100.0
      GGRM              GUDANG GARAM Tbk             100.0


In [51]:
def analyze_free_float(df):
    # Group by issuer and sum the PERCENTAGE column to find total block ownership
    concentration = df.groupby(['SHARE_CODE', 'ISSUER_NAME'])['PERCENTAGE'].sum().reset_index()
    
    # Calculate True Free Float (Assuming anything <1% makes up the free float)
    concentration['ESTIMATED_FREE_FLOAT_%'] = 100 - concentration['PERCENTAGE']
    
    # Sort to find the most concentrated companies (lowest free float)
    most_concentrated = concentration.sort_values(by='ESTIMATED_FREE_FLOAT_%', ascending=True)
    
    print("--- TOP 5 MOST CONCENTRATED STOCKS (Lowest Estimated Free Float) ---")
    print(most_concentrated[['SHARE_CODE', 'ISSUER_NAME', 'PERCENTAGE', 'ESTIMATED_FREE_FLOAT_%']].head().to_string(index=False))

analyze_free_float(df)

--- TOP 5 MOST CONCENTRATED STOCKS (Lowest Estimated Free Float) ---
SHARE_CODE                        ISSUER_NAME  PERCENTAGE  ESTIMATED_FREE_FLOAT_%
      IBST          INTI BANGUN SEJAHTERA Tbk       99.95                    0.05
      PGUN              PRADIKSI GUNATAMA Tbk       99.93                    0.07
      HITS HUMPUSS INTERMODA TRANSPORTASI Tbk       99.92                    0.08
      SUPR           SOLUSI TUNAS PRATAMA Tbk       99.92                    0.08
      DCII                  DCI INDONESIA Tbk       99.85                    0.15


In [54]:
def analyze_hidden_whales(df):
    # Ensure DOMICILE is clean
    df['DOMICILE_CLEAN'] = df['DOMICILE'].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    
    # Isolate foreign entities
    foreign_df = df[~df['DOMICILE_CLEAN'].str.contains('INDONESIA')]
    
    # List of prominent Indonesian business family names / conglomerate keywords to flag
    tycoon_keywords = ['SALIM', 'BAKRIE', 'WIDJAJA', 'HARTONO', 'PANGESTU', 'TANOTO', 'THOHIR', 'PANIGORO']
    
    # Create a regex pattern: 'SALIM|BAKRIE|WIDJAJA...'
    pattern = '|'.join(tycoon_keywords)
    
    # Filter foreign entities containing these names in INVESTOR_NAME
    hidden_whales = foreign_df[foreign_df['INVESTOR_NAME'].str.upper().str.contains(pattern, na=False)]
    
    print("--- POTENTIAL 'HIDDEN' DOMESTIC WHALES (Offshore Entities with Tycoon Names) ---")
    cols_to_show = ['INVESTOR_NAME', 'SHARE_CODE', 'DOMICILE_CLEAN', 'PERCENTAGE']
    print(hidden_whales[cols_to_show].sort_values(by='PERCENTAGE', ascending=False).head(10).to_string(index=False))

analyze_hidden_whales(df)

--- POTENTIAL 'HIDDEN' DOMESTIC WHALES (Offshore Entities with Tycoon Names) ---
               INVESTOR_NAME SHARE_CODE DOMICILE_CLEAN  PERCENTAGE
      SABANA PRAWIRA WIDJAJA       CAMP        UNKNOWN       83.94
 SYA BTbAkNA PRAWIRA WIDJAJA       ULTJ        UNKNOWN       53.17
               PANGESTU ALIM       LMPI        UNKNOWN       30.04
PYR TAbWkIRAWIDJAJA PRAKARSA       ULTJ        UNKNOWN       23.78
            HARTONO HERJANTO       MSJA        UNKNOWN        6.36
               AGUS NURSALIM       KICI        UNKNOWN        4.60
                 OKI WIDJAJA       GLVA        UNKNOWN        2.47
               SURJA HARTONO       SMSM        UNKNOWN        2.26
               MARIA WIDJAJA       BBMD      SINGAPORE        1.55
                EDDY HARTONO       SMSM        UNKNOWN        1.38


In [55]:
import pandas as pd
import os
import glob

# Path to the folder containing your .csv files
folder_path = r'G:\My Drive\00) Interview Prep\00) Quant\Data Sources\WRDS Data\Returns'

# Defined keywords for classification
fundamental_keys = {'at', 'lt', 'sale', 'revt', 'ni', 'ebitda', 'fyear', 'indfmt'}
returns_keys = {'prcc_f', 'prccd', 'prccm', 'trt1m', 'ret', 'retx', 'ajex', 'cshtr'}

def classify_compustat_file(file_path):
    try:
        # Read only the header to save memory/time
        df_headers = pd.read_csv(file_path, nrows=0)
        columns = set(df_headers.columns.str.lower())
        
        if columns.intersection(fundamental_keys):
            return "Fundamental Data"
        elif columns.intersection(returns_keys):
            return "Returns/Price Data"
        else:
            return "Unknown/Other"
            
    except Exception as e:
        return f"Error reading file: {e}"

# Get all CSV files
files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"{'File Name':<40} | {'Classification'}")
print("-" * 60)

for file in files:
    file_name = os.path.basename(file)
    data_type = classify_compustat_file(file)
    print(f"{file_name:<40} | {data_type}")

File Name                                | Classification
------------------------------------------------------------
CRSP_Compustat_LinkingTable.csv          | Unknown/Other
CCM_Quarterly.csv                        | Fundamental Data
CCM_Daily.csv                            | Returns/Price Data
CCM_Annual.csv                           | Fundamental Data
Compustat_Annual.csv                     | Fundamental Data
CCM_Monthly.csv                          | Returns/Price Data
Indexes Market Return.csv                | Unknown/Other
AnalystsPredictions_1970-2025.csv        | Unknown/Other
13f Holdings_Ownership_1978-2025.csv     | Unknown/Other
UNUSED_Global_Security_Master.csv.csv    | Unknown/Other
IBES_Analysts_Summary.csv.csv            | Unknown/Other
